# Analiza per-grupa — zróżnicowanie jakości modelu

## Cel tego notebooka

Sprawdzamy czy model działa jednakowo dobrze dla wszystkich grup demograficznych.

Dzielimy użytkowników na grupy według:
- **Płci** (M/F)
- **Wieku** (7 przedziałów)
- **Zawodu** (21 kategorii)
- **Aktywności** (liczba wystawionych ocen)

Dla każdej grupy mierzymy RMSE i MAE osobno.

Wyniki trafiają bezpośrednio do Rozdziału III —
podrozdział "Zróżnicowanie jakości modelu".

In [1]:
import os
os.chdir(os.path.expanduser('~/magisterka'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid", font_scale=1.1)

lr           = joblib.load('backend/model/linear_model.pkl')
scaler       = joblib.load('backend/model/scaler.pkl')
FEATURE_COLS = joblib.load('backend/model/feature_cols.pkl')

df_model = pd.read_csv('data/df_model.csv')
print(f"Wczytano: {df_model.shape}")
display(df_model.head(3))

Wczytano: (1000209, 48)


,age,gender_encoded,year,user_avg_rating,movie_avg_rating,movie_rating_count,Action,Adventure,Animation,Children's,...,occ_14,occ_15,occ_16,occ_17,occ_18,occ_19,occ_20,rating,userId,movieId
0,1,0,1975.0,4.188679,4.390725,1725,0,0,0,0,...,False,False,False,False,False,False,False,5,1,1193
1,1,0,1996.0,4.188679,3.464762,525,0,0,1,1,...,False,False,False,False,False,False,False,3,1,661
2,1,0,1964.0,4.188679,4.154088,636,0,0,0,0,...,False,False,False,False,False,False,False,3,1,914


## Przygotowanie danych do analizy per-grupa

Wczytujemy oryginalne pliki żeby mieć dostęp do kolumn demograficznych
(płeć, wiek, zawód) które nie są w df_model w czytelnej formie.
Łączymy z przewidywaniami modelu.

In [ ]:
ratings = pd.read_csv('data/ratings.dat', sep='::', engine='python',
                      names=['userId','movieId','rating','timestamp'])
users   = pd.read_csv('data/users.dat', sep='::', engine='python',
                      names=['userId','gender','age','occupation','zip'],
                      encoding='latin-1')
movies  = pd.read_csv('data/movies.dat', sep='::', engine='python',
                      names=['movieId','title','genres'], encoding='latin-1')

# wczytaj X i y
X_all = df_model[FEATURE_COLS].values
y_all = df_model['rating'].values

X_scaled  = scaler.transform(pd.DataFrame(X_all, columns=FEATURE_COLS))
y_pred_all = lr.predict(X_scaled)
y_pred_all = np.clip(y_pred_all, 1.0, 5.0)

# dodaj przewidywania do df_model
df_model['predicted'] = y_pred_all
df_model['residual']  = df_model['rating'] - df_model['predicted']
df_model['abs_error'] = df_model['residual'].abs()

# dodaj dane demograficzne
# usuń kolumny jeśli już istnieją (mogły być w df_model z preprocessingu)
for col in ['gender', 'age', 'occupation']:
    if col in df_model.columns:
        df_model = df_model.drop(columns=[col])

df_model = df_model.merge(users[['userId','gender','age','occupation']], on='userId', how='left')

AGE_LABELS = {1:'<18', 18:'18-24', 25:'25-34', 35:'35-44',
              45:'45-49', 50:'50-55', 56:'56+'}
OCC_LABELS = {0:'other', 1:'educator', 2:'artist', 3:'clerical',
              4:'student', 5:'customer svc', 6:'doctor',
              7:'executive', 8:'farmer', 9:'homemaker', 10:'K-12 student',
              11:'lawyer', 12:'programmer', 13:'retired', 14:'sales',
              15:'scientist', 16:'self-employed', 17:'technician',
              18:'tradesman', 19:'unemployed', 20:'writer'}

df_model['age_label'] = df_model['age'].map(AGE_LABELS)
df_model['occ_label'] = df_model['occupation'].map(OCC_LABELS)

print("Dane przygotowane.")
print(f"Kolumny: {df_model.columns.tolist()}")

KeyError: 'age'

## Analiza 1 — Jakość modelu według płci

Sprawdzamy czy model działa jednakowo dla mężczyzn i kobiet.
MovieLens jest zdominowany przez mężczyzn (~72%) — model może być
mniej dokładny dla kobiet ze względu na niedoreprezentowanie tej grupy.

In [ ]:
gender_stats = df_model.groupby('gender').agg(
    count=('rating', 'count'),
    rmse=('abs_error', lambda x: np.sqrt((x**2).mean())),
    mae=('abs_error', 'mean'),
    avg_real=('rating', 'mean'),
    avg_pred=('predicted', 'mean')
).round(4)
gender_stats.index = ['Kobiety (F)', 'Mężczyźni (M)']
display(gender_stats)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# RMSE
axes[0].bar(gender_stats.index, gender_stats['rmse'],
            color=['#e87040','#4a90d9'], edgecolor='white', width=0.5)
axes[0].set_title('RMSE według płci', fontweight='bold')
axes[0].set_ylabel('RMSE')
for i, (idx, row) in enumerate(gender_stats.iterrows()):
    axes[0].text(i, row['rmse'] + 0.002, f"{row['rmse']:.4f}",
                 ha='center', fontsize=10)

# MAE
axes[1].bar(gender_stats.index, gender_stats['mae'],
            color=['#e87040','#4a90d9'], edgecolor='white', width=0.5)
axes[1].set_title('MAE według płci', fontweight='bold')
axes[1].set_ylabel('MAE')
for i, (idx, row) in enumerate(gender_stats.iterrows()):
    axes[1].text(i, row['mae'] + 0.002, f"{row['mae']:.4f}",
                 ha='center', fontsize=10)

# liczba ocen
axes[2].bar(gender_stats.index, gender_stats['count'],
            color=['#e87040','#4a90d9'], edgecolor='white', width=0.5)
axes[2].set_title('Liczba ocen według płci', fontweight='bold')
axes[2].set_ylabel('Liczba ocen')
for i, (idx, row) in enumerate(gender_stats.iterrows()):
    axes[2].text(i, row['count'] + 1000, f"{int(row['count']):,}",
                 ha='center', fontsize=10)

plt.suptitle('Jakość modelu według płci', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/fig_group_gender.png', dpi=150)
plt.show()

## Analiza 2 — Jakość modelu według wieku

Sprawdzamy czy model działa jednakowo dla różnych grup wiekowych.
Hipoteza: użytkownicy w grupach wiekowych z małą liczbą reprezentantów
będą mieli wyższe RMSE.

In [ ]:
age_order = ['<18','18-24','25-34','35-44','45-49','50-55','56+']

age_stats = df_model.groupby('age_label').agg(
    count=('rating', 'count'),
    rmse=('abs_error', lambda x: np.sqrt((x**2).mean())),
    mae=('abs_error', 'mean'),
    avg_real=('rating', 'mean')
).round(4).reindex(age_order)
display(age_stats)

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

# RMSE i MAE
x = np.arange(len(age_order))
width = 0.35
axes[0].bar(x - width/2, age_stats['rmse'], width,
            label='RMSE', color='#4a90d9', edgecolor='white')
axes[0].bar(x + width/2, age_stats['mae'], width,
            label='MAE', color='#e87040', edgecolor='white')
axes[0].set_title('RMSE i MAE według wieku', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(age_order)
axes[0].set_ylabel('Błąd')
axes[0].legend()
for i, (rmse, mae) in enumerate(zip(age_stats['rmse'], age_stats['mae'])):
    axes[0].text(i - width/2, rmse + 0.002, f'{rmse:.3f}', ha='center', fontsize=8)
    axes[0].text(i + width/2, mae  + 0.002, f'{mae:.3f}',  ha='center', fontsize=8)

# liczba ocen
axes[1].bar(x, age_stats['count'], color='#2ecc71', edgecolor='white')
axes[1].set_title('Liczba ocen według wieku', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(age_order)
axes[1].set_ylabel('Liczba ocen')
for i, val in enumerate(age_stats['count']):
    axes[1].text(i, val + 500, f'{int(val):,}', ha='center', fontsize=9)

plt.suptitle('Jakość modelu według grupy wiekowej',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/fig_group_age.png', dpi=150)
plt.show()

## Analiza 3 — Jakość modelu według zawodu

21 kategorii zawodowych — sprawdzamy zróżnicowanie jakości modelu.
Hipoteza: grupy z małą liczbą reprezentantów (farmer, homemaker)
będą miały wyższe RMSE niż duże grupy (student, programmer).

In [ ]:
occ_stats = df_model.groupby('occ_label').agg(
    count=('rating', 'count'),
    rmse=('abs_error', lambda x: np.sqrt((x**2).mean())),
    mae=('abs_error', 'mean')
).round(4).sort_values('rmse', ascending=True)
display(occ_stats)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#4a90d9' if r < occ_stats['rmse'].median() else '#e87040'
          for r in occ_stats['rmse']]
bars = ax.barh(occ_stats.index, occ_stats['rmse'],
               color=colors, edgecolor='white')
ax.axvline(occ_stats['rmse'].median(), color='red', linestyle='--',
           linewidth=1.5, label=f'Mediana RMSE = {occ_stats["rmse"].median():.4f}')
for bar, val in zip(bars, occ_stats['rmse']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
ax.set_title('RMSE według zawodu użytkownika\n'
             '(niebieski = poniżej mediany, koralowy = powyżej mediany)',
             fontweight='bold')
ax.set_xlabel('RMSE')
ax.legend()
plt.tight_layout()
plt.savefig('notebooks/fig_group_occupation.png', dpi=150)
plt.show()

## Analiza 4 — Jakość modelu według aktywności użytkownika

Dzielimy użytkowników na 4 grupy według liczby wystawionych ocen:
- **Nieaktywni:** < 20 ocen
- **Mało aktywni:** 20–50 ocen
- **Aktywni:** 50–200 ocen
- **Bardzo aktywni:** > 200 ocen

Hipoteza: więcej ocen = lepszy profil użytkownika = niższe RMSE.

In [ ]:
user_activity = df_model.groupby('userId')['rating'].count().rename('n_ratings')
df_model = df_model.merge(user_activity, on='userId')

def activity_group(n):
    if n < 20:   return '1. Nieaktywni (<20)'
    if n < 50:   return '2. Mało aktywni (20-50)'
    if n < 200:  return '3. Aktywni (50-200)'
    return '4. Bardzo aktywni (>200)'

df_model['activity'] = df_model['n_ratings'].apply(activity_group)

activity_stats = df_model.groupby('activity').agg(
    users=('userId', 'nunique'),
    count=('rating', 'count'),
    rmse=('abs_error', lambda x: np.sqrt((x**2).mean())),
    mae=('abs_error', 'mean')
).round(4)
display(activity_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
groups = activity_stats.index.tolist()
x = np.arange(len(groups))

axes[0].bar(x, activity_stats['rmse'], color='#4a90d9', edgecolor='white', width=0.6)
axes[0].set_title('RMSE według aktywności użytkownika', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(groups, rotation=10, ha='right', fontsize=9)
axes[0].set_ylabel('RMSE')
for i, val in enumerate(activity_stats['rmse']):
    axes[0].text(i, val + 0.002, f'{val:.4f}', ha='center', fontsize=9)

axes[1].bar(x, activity_stats['users'], color='#2ecc71', edgecolor='white', width=0.6)
axes[1].set_title('Liczba użytkowników w grupie', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(groups, rotation=10, ha='right', fontsize=9)
axes[1].set_ylabel('Liczba użytkowników')
for i, val in enumerate(activity_stats['users']):
    axes[1].text(i, val + 5, f'{int(val):,}', ha='center', fontsize=9)

plt.suptitle('Wpływ aktywności użytkownika na jakość predykcji',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/fig_group_activity.png', dpi=150)
plt.show()

## Podsumowanie analizy per-grupa

Zestawienie wszystkich grup w jednej tabeli — gotowe do wklejenia w pracę.

In [ ]:
print("=" * 55)
print("PODSUMOWANIE — RMSE PER GRUPA")
print("=" * 55)

print("\n📊 Według płci:")
display(gender_stats[['count','rmse','mae']])

print("\n📊 Według wieku:")
display(age_stats[['count','rmse','mae']])

print("\n📊 Według aktywności:")
display(activity_stats[['users','count','rmse','mae']])

print("\n📊 Najlepsze i najgorsze zawody (RMSE):")
display(pd.concat([
    occ_stats.head(5)[['count','rmse','mae']],
    pd.DataFrame([['...','...','...']], columns=['count','rmse','mae'],
                 index=['---']),
    occ_stats.tail(5)[['count','rmse','mae']]
]))

# zapis do pliku
summary = {
    'gender':     gender_stats[['rmse','mae']].to_dict(),
    'age':        age_stats[['rmse','mae']].to_dict(),
    'activity':   activity_stats[['rmse','mae']].to_dict(),
    'occupation': occ_stats[['rmse','mae']].to_dict()
}
import json
with open('data/group_analysis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print("\nZapisano: data/group_analysis_summary.json")